In [ ]:
!pip install anthropic

In [ ]:
import os
import uuid
import json
import time
import subprocess
import hashlib
import threading
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Callable
from datetime import datetime
import anthropic
import glob as glob_module

os.environ["ANTHROPIC_API_KEY"] = ""
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

# 1) Agent Loop

This is the beating heart of every LLM agent. Claude Code, Codex CLI,
Cursor — they all share this identical pattern. Everything else in this
notebook is "harness": infrastructure that surrounds this loop.

The loop is simple:
  1. Send messages to the LLM (with tool definitions)
  2. If the model says stop_reason == "end_turn" → return the text
  3. If the model says stop_reason == "tool_use" → execute the tool(s),
     append results, and loop back to step 1

The MODEL decides when to call tools and when to stop.
The CODE just executes what the model asks for.

This is the fundamental insight: the agent IS the model. The code is
the harness — it gives the model hands, eyes, and a workspace.

Architecture (from the Claude Code source):

```
  User ──► messages[] ──► LLM ──► response
                                     │
                           stop_reason == "tool_use"?
                          /                          \
                        yes                           no
                         │                             │
                   execute tools                  return text
                   append results                 (done)
                   loop back ──────────────► messages[]
```

In [ ]:
def agent_loop(
    messages: list[dict],
    system: str = "",
    tools: list[dict] = None,
    tool_handlers: dict[str, Callable] = None,
    max_iterations: int = 20,
    on_tool_call: Callable = None,     # hook: called before each tool execution
    on_response: Callable = None,      # hook: called on each LLM response
) -> list[dict]:
    """
    The core agent loop. This is functionally equivalent to the main loop
    in Claude Code's query.ts (the largest file at 785KB in the leaked source).

    Args:
        messages:       Conversation history — list of {role, content} dicts.
                        The loop MUTATES this list (appends assistant + tool results).
        system:         System prompt string.
        tools:          List of tool JSON schemas (Anthropic format).
        tool_handlers:  Dict mapping tool_name → callable(**input) → str.
        max_iterations: Safety limit to prevent infinite loops.
        on_tool_call:   Optional callback(tool_name, tool_input) before execution.
        on_response:    Optional callback(response) after each LLM call.

    Returns:
        The mutated messages list (with full conversation history).
    """
    tools = tools or []
    tool_handlers = tool_handlers or {}

    for iteration in range(max_iterations):
        # ── Step 1: Call the LLM ──────────────────────────────────────
        # This is the ONLY place we talk to the API. Everything else is
        # tool execution and message bookkeeping.
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=system,
            messages=messages,
            tools=tools if tools else anthropic.NOT_GIVEN,
        )

        if on_response:
            on_response(response)

        # ── Step 2: Append the assistant's response to history ────────
        # We ALWAYS append the full response, whether it's text or tool_use.
        # This maintains the conversation state the model sees on next turn.
        messages.append({
            "role": "assistant",
            "content": response.content,
        })

        # ── Step 3: Check stop reason ─────────────────────────────────
        # "end_turn" = model is done talking, return.
        # "tool_use" = model wants to call one or more tools.
        # "max_tokens" = response was truncated (rare with 4096).
        if response.stop_reason != "tool_use":
            # Extract final text for convenience
            return messages

        # ── Step 4: Execute ALL tool calls in this response ───────────
        # The model can request MULTIPLE tools in a single response.
        # We must execute them all and return results in one user message.
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue

            tool_name = block.name
            tool_input = block.input
            tool_use_id = block.id

            if on_tool_call:
                on_tool_call(tool_name, tool_input)

            # Dispatch to the handler (Function Calling)
            if tool_name in tool_handlers:
                try:
                    output = tool_handlers[tool_name](**tool_input)
                except Exception as e:
                    output = f"Error executing {tool_name}: {type(e).__name__}: {e}"
            else:
                output = f"Unknown tool: {tool_name}"

            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tool_use_id,
                "content": str(output),
            })

        # ── Step 5: Append tool results and loop back ─────────────────
        # Tool results are sent as a "user" message. This is the Anthropic
        # API convention — tool results come from the user role.
        messages.append({
            "role": "user",
            "content": tool_results,
        })

    # Safety: if we hit max_iterations, return what we have
    print(f"⚠️  Agent hit max_iterations={max_iterations}")
    return messages

In [ ]:
# ── Quick Demo: Bare-bones agent with one tool ───────────────────────────

def _demo_bash_tool(command: str) -> str:
    """Execute a shell command and return stdout+stderr."""
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True, timeout=10
        )
        output = result.stdout + result.stderr
        return output.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: command timed out (10s)"

DEMO_TOOLS = [{
    "name": "bash",
    "description": "Execute a shell command. Use this for file operations, running code, git, etc.",
    "input_schema": {
        "type": "object",
        "properties": {
            "command": {
                "type": "string",
                "description": "The shell command to execute."
            }
        },
        "required": ["command"]
    }
}]

DEMO_HANDLERS = {"bash": _demo_bash_tool}

print("=" * 70)
print("DEMO — Agent Loop with Bash Tool")
print("=" * 70)

demo_messages = [{"role": "user", "content": "What Python version is installed? Use the bash tool."}]
result = agent_loop(
    messages=demo_messages,
    system="You are a helpful coding assistant. Use tools when needed.",
    tools=DEMO_TOOLS,
    tool_handlers=DEMO_HANDLERS,
    on_tool_call=lambda name, inp: print(f"  🔧 Tool call: {name}({inp})"),
)
# Print the final assistant message
for block in result[-1]["content"]:
    if hasattr(block, "text"):
        print(f"  🤖 Agent: {block.text}")

DEMO — Agent Loop with Bash Tool
  🔧 Tool call: bash({'command': 'python --version'})
  🤖 Agent: The installed Python version is **Python 3.12.13**.


# 2) TOOL SYSTEM

Claude Code has 40+ tools. Each tool is:
  1. A JSON schema (name, description, input_schema) — sent to the LLM
  2. A handler function — executed locally when the model calls it

The tool list in Claude Code (from the leaked source):
  * Core:     BashTool, FileReadTool, FileWriteTool, FileEditTool
  * Search:   GlobTool, GrepTool, ToolSearchTool
  * Web:      WebFetchTool, WebSearchTool
  * Agent:    AgentTool (subagent spawning — covered in §5)
  * Planning: TodoWriteTool, TodoReadTool (covered in §6)
  * System:   ConfigTool, StatusTool, ExitTool
  * Advanced: SkillTool, MCPTool, NotebookTool, REPLTool
  * Teams:    TeamTool, TaskTool (covered in §10)

> Key design principle: each tool is ATOMIC and COMPOSABLE. The model
decides the orchestration — the harness just provides primitives. The tool schemas and handler implementations for the core tools are in: `utils/tool_definitions.py`

## 2.1 Bash

The "crown jewel" of Claude Code. A general-purpose shell executor.
In Claude Code, BashTool handles:
  - Running code (python, node, cargo, etc.)
  - Git operations (commit, push, diff, log)
  - Package management (pip, npm, cargo)
  - System inspection (ls, cat, head, wc)
  - ANYTHING the model wants to do on the system

The model is effectively given root shell access. The permission model
(§4) is what gates dangerous operations.

In [ ]:
def bash_tool(command: str, timeout: int = 30) -> str:
    """
    Execute a shell command. Returns combined stdout + stderr.

    In Claude Code's BashTool implementation, there are additional features:
    - Working directory tracking (cd persists across calls via shell state)
    - Background process support (commands ending with &)
    - Process group management for cleanup
    - Output truncation for very long outputs
    - The shell session is persistent (not a fresh subprocess each time)
    """
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd=os.getcwd(),
        )
        output = result.stdout
        if result.stderr:
            output += "\n[stderr]\n" + result.stderr
        # Truncate very long outputs (Claude Code does this too)
        if len(output) > 10000:
            output = output[:5000] + "\n\n... [truncated] ...\n\n" + output[-5000:]
        return output.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return f"Error: command timed out after {timeout}s"
    except Exception as e:
        return f"Error: {e}"

## 2.2 FileRead

Reads file contents. In Claude Code this supports:
  - Line range selection (start_line, end_line)
  - Binary file detection (returns "Binary file" message)
  - Encoding detection
  - Max file size limits

In [ ]:
def file_read_tool(file_path: str, start_line: int = None, end_line: int = None) -> str:
    """Read a file's contents, optionally a specific line range."""
    try:
        path = Path(file_path).resolve()
        if not path.exists():
            return f"Error: file not found: {file_path}"
        if not path.is_file():
            return f"Error: not a file: {file_path}"

        text = path.read_text(encoding="utf-8", errors="replace")
        lines = text.splitlines(keepends=True)

        if start_line is not None or end_line is not None:
            start = (start_line or 1) - 1  # 1-indexed to 0-indexed
            end = end_line or len(lines)
            lines = lines[start:end]

        content = "".join(lines)
        if len(content) > 50000:
            content = content[:50000] + "\n... [file truncated at 50KB]"
        return content
    except Exception as e:
        return f"Error reading {file_path}: {e}"

## 2.3 File Write

Creates or overwrites a file.

Claude Code also has FileEditTool which
does surgical edits (search/replace within a file), but write covers
the basic case.

In [ ]:
def file_write_tool(file_path: str, content: str) -> str:
    """Write content to a file. Creates parent directories if needed."""
    try:
        path = Path(file_path).resolve()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
        return f"Successfully wrote {len(content)} bytes to {file_path}"
    except Exception as e:
        return f"Error writing {file_path}: {e}"

## 2.4 GlobTool

In [ ]:
def glob_tool(pattern: str, path: str = ".") -> str:
    """Find files matching a glob pattern."""
    try:
        matches = sorted(glob_module.glob(
            os.path.join(path, pattern), recursive=True
        ))
        if not matches:
            return "No files matched."
        return "\n".join(matches[:100])  # cap at 100 results
    except Exception as e:
        return f"Error: {e}"

## 2.5 GrepTool

Searches file contents. Critical for "find where X is defined" queries.

In [ ]:
def grep_tool(pattern: str, path: str = ".", include: str = None) -> str:
    """Search for a pattern in files (like ripgrep)."""
    try:
        cmd = f"grep -rn '{pattern}' {path}"
        if include:
            cmd += f" --include='{include}'"
        result = subprocess.run(
            cmd, shell=True, capture_output=True, text=True, timeout=10
        )
        output = result.stdout.strip()
        if not output:
            return f"No matches for '{pattern}'"
        # Limit output
        lines = output.split("\n")
        if len(lines) > 50:
            output = "\n".join(lines[:50]) + f"\n... [{len(lines)-50} more matches]"
        return output
    except Exception as e:
        return f"Error: {e}"

## 2.6 Tool registry

In [ ]:
CORE_TOOL_SCHEMAS = [
    {
        "name": "bash",
        "description": (
            "Execute a shell command. Use for running code, git operations, "
            "package management, file operations, and system commands. "
            "The working directory is the project root."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "The shell command to run."},
                "timeout": {"type": "integer", "description": "Timeout in seconds (default: 30)."},
            },
            "required": ["command"],
        },
    },
    {
        "name": "file_read",
        "description": "Read a file's contents. Supports optional line range selection.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path to the file to read."},
                "start_line": {"type": "integer", "description": "Start line (1-indexed, optional)."},
                "end_line": {"type": "integer", "description": "End line (inclusive, optional)."},
            },
            "required": ["file_path"],
        },
    },
    {
        "name": "file_write",
        "description": "Write content to a file. Creates the file and parent dirs if they don't exist.",
        "input_schema": {
            "type": "object",
            "properties": {
                "file_path": {"type": "string", "description": "Path to write to."},
                "content": {"type": "string", "description": "The full content to write."},
            },
            "required": ["file_path", "content"],
        },
    },
    {
        "name": "glob",
        "description": "Find files matching a glob pattern (e.g. '**/*.py'). Useful for navigating codebases.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Glob pattern (supports **)"},
                "path": {"type": "string", "description": "Base directory (default: '.')"},
            },
            "required": ["pattern"],
        },
    },
    {
        "name": "grep",
        "description": "Search for a text pattern across files (like ripgrep). Returns matching lines with file paths and line numbers.",
        "input_schema": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Regex pattern to search for."},
                "path": {"type": "string", "description": "Directory to search in (default: '.')."},
                "include": {"type": "string", "description": "File glob to filter (e.g. '*.py')."},
            },
            "required": ["pattern"],
        },
    },
]

CORE_TOOL_HANDLERS = {
    "bash": bash_tool,
    "file_read": file_read_tool,
    "file_write": file_write_tool,
    "glob": glob_tool,
    "grep": grep_tool,
}

print(f"Registered {len(CORE_TOOL_SCHEMAS)} core tools: {[t['name'] for t in CORE_TOOL_SCHEMAS]}")

Registered 5 core tools: ['bash', 'file_read', 'file_write', 'glob', 'grep']


# 3) System Design

## 3.1 System Prompt

The system prompt is what gives the agent its personality, knowledge,
and instructions. In Claude Code, the system prompt is ASSEMBLED from
multiple sources at startup — it's not a static string.

Claude Code's prompt assembly pipeline (from the leaked source):

  1. BASE PROMPT — hard-coded persona, capabilities, guidelines
     (lives in src/ as TypeScript string constants)

  2. CLAUDE.md DISCOVERY — searches for instruction files:
     - ~/.claude/CLAUDE.md          (user-global preferences)
     - ~/project/CLAUDE.md          (project-level instructions)
     - ~/project/subdir/CLAUDE.md   (directory-level overrides)
     These are merged in order (global → project → local).
     This is the "self-healing memory" system.

  3. CONTEXT INJECTION — dynamic context added per-session:
     - Current working directory
     - Git status / branch info
     - Available tools list
     - Permission mode
     - Active skills (from SKILL.md files)

  4. CACHED vs UNCACHED SECTIONS — some sections are marked as
     "cache-safe" (they don't change between turns) and some are
     "uncached" (DANGEROUS_uncachedSystemPromptSection). This is
     critical for prompt cache economics at scale.

The full prompt templates are in: utils/system_prompts.py

In [ ]:
def discover_claude_md(project_root: str = ".") -> list[dict]:
    """
    Discover CLAUDE.md files in the hierarchy, mimicking Claude Code's
    CLAUDE.md discovery system.

    Search order (all merged into the system prompt):
      1. ~/.claude/CLAUDE.md         — user-global preferences
      2. {project_root}/CLAUDE.md    — project instructions
      3. Any subdirectory CLAUDE.md  — scoped overrides

    Returns:
        List of {"path": str, "content": str} dicts, ordered global → local.
    """
    found = []

    # User-global CLAUDE.md
    global_path = Path.home() / ".claude" / "CLAUDE.md"
    if global_path.exists():
        found.append({
            "path": str(global_path),
            "content": global_path.read_text(encoding="utf-8", errors="replace"),
            "scope": "global",
        })

    # Project-level CLAUDE.md
    project_path = Path(project_root).resolve() / "CLAUDE.md"
    if project_path.exists():
        found.append({
            "path": str(project_path),
            "content": project_path.read_text(encoding="utf-8", errors="replace"),
            "scope": "project",
        })

    return found

In [ ]:
def get_dynamic_context(project_root: str = ".") -> str:
    """
    Gather dynamic context about the current environment.
    In Claude Code, this is injected into the system prompt on every session.
    """
    context_parts = []

    # Working directory
    context_parts.append(f"Working directory: {os.path.abspath(project_root)}")

    # Git branch (if in a git repo)
    try:
        branch = subprocess.run(
            "git branch --show-current", shell=True,
            capture_output=True, text=True, timeout=5
        ).stdout.strip()
        if branch:
            context_parts.append(f"Git branch: {branch}")
    except Exception:
        pass

    # Timestamp
    context_parts.append(f"Current time: {datetime.now().isoformat()}")

    # Platform
    import platform
    context_parts.append(f"Platform: {platform.system()} {platform.machine()}")

    return "\n".join(context_parts)

In [ ]:
def assemble_system_prompt(
    project_root: str = ".",
    permission_mode: str = "normal",
    extra_instructions: str = "",
) -> str:
    """
    Assemble the full system prompt from all sources.

    This mirrors Claude Code's prompt assembly pipeline.
    The full base prompt template is in utils/system_prompts.py.
    """
    sections = []

    # ── Section 1: Base persona ───────────────────────────────────────
    # In Claude Code, this is a ~2000 token block that establishes:
    # - The agent's identity and capabilities
    # - How to use tools effectively
    # - Code style guidelines
    # - Error handling philosophy
    base_prompt = (
        "You are an expert software engineer acting as a coding agent. "
        "You have access to tools for reading files, writing files, "
        "running shell commands, and searching codebases.\n\n"
        "GUIDELINES:\n"
        "- Always read existing code before modifying it.\n"
        "- Use grep/glob to understand the codebase structure first.\n"
        "- Make minimal, targeted changes. Don't rewrite entire files.\n"
        "- Run tests after making changes to verify correctness.\n"
        "- If a task is complex, break it into subtasks and use the "
        "todo system to track progress.\n"
        "- Explain your reasoning before taking actions.\n"
        "- If you're unsure, ask the user for clarification.\n"
    )
    sections.append(base_prompt)

    # ── Section 2: CLAUDE.md instructions ─────────────────────────────
    claude_md_files = discover_claude_md(project_root)
    if claude_md_files:
        sections.append("# Project Instructions (from CLAUDE.md files)")
        for entry in claude_md_files:
            sections.append(f"\n## [{entry['scope']}] {entry['path']}\n{entry['content']}")

    # ── Section 3: Dynamic context ────────────────────────────────────
    sections.append(f"\n# Environment\n{get_dynamic_context(project_root)}")

    # ── Section 4: Permission mode ────────────────────────────────────
    sections.append(f"\n# Permission Mode: {permission_mode}")
    if permission_mode == "ask":
        sections.append(
            "You must ask the user for confirmation before running any "
            "destructive commands (rm, git push, etc.) or writing files."
        )

    # ── Section 5: Extra instructions ─────────────────────────────────
    if extra_instructions:
        sections.append(f"\n# Additional Instructions\n{extra_instructions}")

    return "\n\n".join(sections)

In [ ]:
# Demo: build and inspect a system prompt
demo_prompt = assemble_system_prompt(permission_mode="normal")
print(f"Assembled system prompt ({len(demo_prompt)} chars):")
print(demo_prompt[:500] + "\n...")

Assembled system prompt (736 chars):
You are an expert software engineer acting as a coding agent. You have access to tools for reading files, writing files, running shell commands, and searching codebases.

GUIDELINES:
- Always read existing code before modifying it.
- Use grep/glob to understand the codebase structure first.
- Make minimal, targeted changes. Don't rewrite entire files.
- Run tests after making changes to verify correctness.
- If a task is complex, break it into subtasks and use the todo system to track progress.

...


## 3.2 Permission Model

**Demo: build and inspect a system prompt**

Claude Code has 3 permission modes:

  1. "ask" (default)      — asks user before running bash, writing files
  2. "auto"               — auto-approve most operations, ask for dangerous ones
  3. "dangerFullAccess"   — approve everything (used by power users)

On top of the mode, there's a per-tool policy engine:
  - deny_tools: list of tool names to completely block
  - deny_prefixes: list of command prefixes to block (e.g. "rm -rf /")
  - allowlist: specific commands that are always allowed

The permission check happens BETWEEN the model requesting a tool call
and the harness executing it. This is the on_tool_call hook in our loop.

In Claude Code, permissions are also tied to:
  - Hooks (PreToolUse / PostToolUse) — user-defined scripts that can
    approve, deny, or modify tool calls
  - MCP tool policies — per-server tool restrictions

In [ ]:
@dataclass
class PermissionPolicy:
    """
    Permission configuration for the agent.

    In Claude Code, this is configured via:
      - CLI flags: --dangerously-skip-permissions
      - .claude.json: {"permissions": {"mode": "auto", "deny": [...]}}
      - CLAUDE.md: can reference permission preferences
    """
    # "ask", "auto", "dangerFullAccess"
    mode: str = "auto"

    # Tools that are completely blocked
    deny_tools: list[str] = field(default_factory=list)

    # Command prefixes that are blocked in bash (safety rails)
    deny_command_prefixes: list[str] = field(default_factory=lambda: [
        "rm -rf /",
        "sudo rm",
        "mkfs",
        "> /dev/sd",
        "dd if=",
    ])

    # Commands that are always allowed without asking (even in "ask" mode)
    auto_approve_commands: list[str] = field(default_factory=lambda: [
        "ls", "cat", "head", "tail", "wc", "echo", "pwd", "date",
        "git status", "git log", "git diff", "git branch",
        "python --version", "node --version",
    ])

    def check_tool(self, tool_name: str, tool_input: dict) -> tuple[bool, str]:
        """
        Check if a tool call is allowed.

        Returns:
            (allowed: bool, reason: str)
        """

        # "dangerFullAccess" mode: approve everything
        if self.mode == "dangerFullAccess":
            return True, "Approved (dangerFullAccess mode)."

        # Check deny list
        if tool_name in self.deny_tools:
            return False, f"Tool '{tool_name}' is in the deny list."

        # For bash commands, check command-level policies
        if tool_name == "bash" and "command" in tool_input:
            cmd = tool_input["command"]

            # Check deny prefixes
            for prefix in self.deny_command_prefixes:
                if cmd.strip().startswith(prefix):
                    return False, f"Command blocked by deny prefix: '{prefix}'"

            # In "ask" mode, auto-approve safe commands
            if self.mode == "ask":
                for safe_cmd in self.auto_approve_commands:
                    if cmd.strip().startswith(safe_cmd):
                        return True, "Auto-approved (safe command)."
                # Everything else needs user approval in "ask" mode
                return False, "Requires user approval (ask mode)."

        return True, "Approved."

def make_permission_hook(policy: PermissionPolicy):
    """
    Create an on_tool_call hook that enforces permissions.

    In Claude Code, this logic lives in toolExecution.ts and toolHooks.ts.
    The hook can:
      - Allow the call (do nothing)
      - Block the call (raise an exception that gets caught)
      - Modify the call (rewrite the input)
      - Ask the user for confirmation (interactive)
    """
    def hook(tool_name: str, tool_input: dict):
        allowed, reason = policy.check_tool(tool_name, tool_input)
        if not allowed:
            print(f"  🚫 BLOCKED: {tool_name} — {reason}")
            # In a real implementation, you'd ask the user
        else:
            print(f"  ✅ {tool_name} — {reason}")
    return hook

In [ ]:
# Demo
policy = PermissionPolicy(mode="ask")
print("Testing permission checks:")
print("  ls:", policy.check_tool("bash", {"command": "ls -la"}))
print("  rm:", policy.check_tool("bash", {"command": "rm -rf /"}))
print("  git:", policy.check_tool("bash", {"command": "git status"}))
print("  write:", policy.check_tool("file_write", {"file_path": "test.py"}))

Testing permission checks:
  ls: (True, 'Auto-approved (safe command).')
  rm: (False, "Command blocked by deny prefix: 'rm -rf /'")
  git: (True, 'Auto-approved (safe command).')
  write: (True, 'Approved.')


## 3.3 Subagent

This is one of Claude Code's most powerful features. The main agent can
spawn "subagents" — child agents that get:
  - Their own fresh messages [] (clean context, no parent noise)
  - A scoped task description (focused on one subtask)
  - The same tool set (or a restricted subset)
  - Their own iteration budget

The subagent runs to completion, then its final output is returned
to the parent agent as a tool result.

Why this matters:
  - Context isolation: the child doesn't pollute the parent's context
  - Parallelism potential: multiple subagents can run concurrently
  - Specialization: each subagent gets instructions tailored to its task

In Claude Code's source, this is AgentTool in src/tools/AgentTool.ts.
The coordinator mode manages multiple agents.

Implementation: it's literally a RECURSIVE CALL to the same agent loop
with a new messages list and a task-specific system prompt.

In [ ]:
def agent_tool(
    task: str,
    parent_system: str = "",
    tools: list[dict] = None,
    tool_handlers: dict[str, Callable] = None,
    max_iterations: int = 10,
) -> str:
    """
    Spawn a subagent to handle a focused task.

    This is the AgentTool implementation. The parent agent calls this
    when it wants to delegate a subtask to a fresh context.

    In Claude Code's AgentTool, additional features include:
    - Task-specific system prompt augmentation
    - Result summarization (if the subagent's output is too long)
    - Error isolation (subagent failures don't crash the parent)
    - Concurrent execution (multiple subagents in parallel)

    Args:
        task: Natural language description of the task.
        parent_system: The parent's system prompt (inherited by child).
        tools: Tool schemas available to the subagent.
        tool_handlers: Tool handler functions.
        max_iterations: Max tool-call iterations for the subagent.

    Returns:
        The subagent's final text response.
    """
    # Build a focused system prompt for the subagent
    subagent_system = (
        f"{parent_system}\n\n"
        f"# Subagent Task\n"
        f"You are a focused subagent. Complete the following task and report "
        f"your results. Be thorough but concise.\n\n"
        f"Task: {task}"
    )

    # Fresh messages — this is the key: clean context
    subagent_messages = [{"role": "user", "content": task}]

    # Run the agent loop recursively
    try:
        result_messages = agent_loop(
            messages=subagent_messages,
            system=subagent_system,
            tools=tools or [],
            tool_handlers=tool_handlers or {},
            max_iterations=max_iterations,
        )

        # Extract the final assistant text
        if result_messages:
            last = result_messages[-1]
            if last["role"] == "assistant":
                texts = []
                for block in last["content"]:
                    if hasattr(block, "text"):
                        texts.append(block.text)
                return "\n".join(texts) if texts else "(subagent produced no text output)"

        return "(subagent produced no output)"

    except Exception as e:
        return f"Subagent error: {type(e).__name__}: {e}"

In [ ]:
AGENT_TOOL_SCHEMA = {
    "name": "agent",
    "description": (
        "Spawn a subagent to handle a focused subtask. The subagent gets "
        "a fresh context and the same tools. Use this for tasks that would "
        "benefit from isolated, focused execution — like investigating a "
        "bug in a specific file while you continue working on the main task."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "task": {
                "type": "string",
                "description": "A clear, self-contained description of the subtask."
            }
        },
        "required": ["task"]
    }
}

AgentTool registered — subagent spawning is ready.
The parent agent can now call agent(task='...') to delegate work.



## 3.4 TodoWrite & Planning

Before executing, effective agents PLAN. Claude Code uses a TodoWrite
tool that creates a structured checklist. The model:
  1. Breaks the task into subtasks
  2. Writes them to a todo list (in memory or on disk)
  3. Executes subtasks one-by-one, checking them off
  4. A "nag reminder" periodically reminds the model to check the list

The nag reminder is a clever trick: after every N tool calls, the harness
injects a system message saying

> "Remember to check your todo list and update task status."

This prevents the model from going off-track.

In Claude Code, the todo state is stored in-memory and exposed via
TodoReadTool and TodoWriteTool.

In [ ]:
@dataclass
class TodoItem:
    id: str
    description: str
    status: str = "pending"  # pending, in_progress, done, blocked

In [ ]:
class TodoManager:
    """
    In-memory todo list for structured planning.

    In Claude Code, this is backed by an in-memory store that persists
    across tool calls within a session. The model can create, update,
    and query tasks through TodoWriteTool and TodoReadTool.
    """
    def __init__(self):
        self._items: dict[str, TodoItem] = {}
        self._counter = 0

    def add(self, description: str) -> TodoItem:
        self._counter += 1
        item = TodoItem(id=f"todo_{self._counter}", description=description)
        self._items[item.id] = item
        return item

    def update(self, todo_id: str, status: str) -> str:
        if todo_id not in self._items:
            return f"Error: todo '{todo_id}' not found."
        self._items[todo_id].status = status
        return f"Updated {todo_id} → {status}"

    def list_all(self) -> str:
        if not self._items:
            return "No todos."
        lines = []
        for item in self._items.values():
            icon = {"pending": "⬜", "in_progress": "🔄", "done": "✅", "blocked": "🚫"}
            lines.append(f"  {icon.get(item.status, '?')} [{item.id}] {item.description} ({item.status})")
        return "\n".join(lines)

    def get_nag_reminder(self, tool_call_count: int, interval: int = 5) -> str | None:
        """
        Generate a nag reminder every N tool calls.
        Returns None if no reminder is needed.

        In Claude Code, this is injected as a system-level message
        into the conversation to keep the model on track.
        """
        if tool_call_count % interval != 0 or tool_call_count == 0:
            return None

        pending = [i for i in self._items.values() if i.status in ("pending", "in_progress")]
        if not pending:
            return None

        return (
            f"[System Reminder] You have {len(pending)} pending/in-progress tasks. "
            f"Review your todo list and update task statuses as you complete them.\n"
            f"Current todos:\n{self.list_all()}"
        )

In [ ]:
todo_manager = TodoManager()

def todo_write_handler(action: str, description: str = "", todo_id: str = "", status: str = "") -> str:
    if action == "add":
        item = todo_manager.add(description)
        return f"Created {item.id}: {item.description}"
    elif action == "update":
        return todo_manager.update(todo_id, status)
    elif action == "list":
        return todo_manager.list_all()
    return f"Unknown action: {action}"

TODO_TOOL_SCHEMA = {
    "name": "todo",
    "description": (
        "Manage a todo list for tracking subtasks. Use this to plan "
        "complex work BEFORE executing. Actions: 'add' (create new task), "
        "'update' (change status), 'list' (show all tasks)."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "action": {"type": "string", "enum": ["add", "update", "list"]},
            "description": {"type": "string", "description": "Task description (for 'add')."},
            "todo_id": {"type": "string", "description": "Task ID (for 'update')."},
            "status": {"type": "string", "enum": ["pending", "in_progress", "done", "blocked"]},
        },
        "required": ["action"],
    },
}

# Demo
todo_manager.add("Read the existing codebase")
todo_manager.add("Identify the bug in parser.py")
todo_manager.add("Write a fix and add tests")
print("Todo list:")
print(todo_manager.list_all())
print("\nNag reminder (at tool call #5):", todo_manager.get_nag_reminder(5))

Todo list:
  ⬜ [todo_1] Read the existing codebase (pending)
  ⬜ [todo_2] Identify the bug in parser.py (pending)
  ⬜ [todo_3] Write a fix and add tests (pending)

Nag reminder (at tool call #5): [System Reminder] You have 3 pending/in-progress tasks. Review your todo list and update task statuses as you complete them.
Current todos:
  ⬜ [todo_1] Read the existing codebase (pending)
  ⬜ [todo_2] Identify the bug in parser.py (pending)
  ⬜ [todo_3] Write a fix and add tests (pending)


## 3.5 Context Compaction

As conversations grow, they eventually exceed the model's context window.

Claude Code uses a 3-layer compression strategy:

  1. Layer 1: TRUNCATION — drop old tool outputs (keep tool calls, drop results)
  2. Layer 2: SUMMARIZATION — ask the model to summarize the conversation so far
  3. Layer 3: HARD RESET — keep only the system prompt + summary + last N messages

The compaction is triggered when the conversation exceeds a token threshold
(typically ~80% of the context window).

In Claude Code, this is handled by autoCompact.ts, which also has a safety mechanism: after 3 consecutive compaction failures, it stops trying (a lesson learned from production where
sessions had 3,000+ consecutive failures wasting 250K API calls/day).

The summary itself becomes a "compressed memory" that preserves the essential context without the verbatim history.

In [ ]:
def estimate_tokens(messages: list[dict]) -> int:
    """
    Rough token estimate. In production, use tiktoken or the API's
    token counting endpoint. Rule of thumb: ~4 chars per token.
    """
    total_chars = 0
    for msg in messages:
        if isinstance(msg.get("content"), str):
            total_chars += len(msg["content"])
        elif isinstance(msg.get("content"), list):
            for block in msg["content"]:
                if isinstance(block, dict):
                    total_chars += len(json.dumps(block))
                elif hasattr(block, "text"):
                    total_chars += len(block.text)
    return total_chars // 4

def compact_messages(
    messages: list[dict],
    system: str,
    max_tokens: int = 80000,
    keep_last_n: int = 4,
) -> list[dict]:
    """
    3-layer context compaction.

    In Claude Code (autoCompact.ts), this is triggered automatically
    when the conversation approaches the context window limit.

    Layer 1: Truncate old tool results (keep the tool call, replace
             the result with "[output truncated]")
    Layer 2: Summarize the conversation history into a compact form
    Layer 3: Keep only system + summary + last N messages

    Args:
        messages: The full conversation history.
        system: System prompt (needed for summarization call).
        max_tokens: Target max tokens after compaction.
        keep_last_n: Number of recent messages to always preserve.

    Returns:
        Compacted messages list.
    """
    current_tokens = estimate_tokens(messages)
    if current_tokens <= max_tokens:
        return messages  # No compaction needed

    print(f"  📦 Compacting: {current_tokens} tokens → target {max_tokens}")

    # ── Layer 1: Truncate old tool outputs ────────────────────────────
    # Keep the most recent messages intact, truncate older tool results
    compacted = []
    preserved_start = max(0, len(messages) - keep_last_n)

    for i, msg in enumerate(messages):
        if i >= preserved_start:
            # Keep recent messages intact
            compacted.append(msg)
        elif msg["role"] == "user" and isinstance(msg.get("content"), list):
            # This is a tool_result message — truncate the output
            truncated_results = []
            for block in msg["content"]:
                if isinstance(block, dict) and block.get("type") == "tool_result":
                    # content got overwritten
                    truncated_results.append({
                        **block,
                        "content": "[output truncated during compaction]",
                    })
                else:
                    truncated_results.append(block)
            # content got overwritten
            compacted.append({**msg, "content": truncated_results})
        else:
            compacted.append(msg)

    # ── Layer 2: Summarize if still over budget ───────────────────────
    if estimate_tokens(compacted) > max_tokens:
        # Ask the model to summarize the old part of the conversation
        old_messages = compacted[:preserved_start]
        old_text = json.dumps(old_messages, indent=2, default=str)[:8000]

        summary_response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            system="Summarize this conversation history in 2-3 paragraphs. Focus on: what was accomplished, what decisions were made, and what's still pending.",
            messages=[{"role": "user", "content": f"Conversation to summarize:\n{old_text}"}],
        )
        summary = summary_response.content[0].text

        # ── Layer 3: Hard reset with summary ──────────────────────────
        compacted = [
            {"role": "user", "content": f"[Conversation summary from earlier in this session]\n{summary}"},
            {"role": "assistant", "content": "Understood. I'll continue from where we left off based on this context."},
        ] + compacted[preserved_start:]

    new_tokens = estimate_tokens(compacted)
    print(f"  📦 Compacted: {current_tokens} → {new_tokens} tokens")
    return compacted

## 3.6 Multi Agent


Claude Code supports spawning TEAMS of agents for complex tasks.
The architecture:

```
  COORDINATOR (lead agent)
    ├── Worker Agent A — "investigate the auth bug"
    ├── Worker Agent B — "write tests for the parser"
    └── Worker Agent C — "update the documentation"
```

Key insight from the leaked source

> The orchestration algorithm is a prompt, not code

The coordinator agent manages workers through system prompt instructions, not hard-coded logic.

From coordinator Mode.ts:
  - "Do not rubber-stamp weak work"
  - "You must understand findings before directing follow-up work"
  - "Never hand off understanding to another worker"

Communication uses JSONL mailboxes — each agent has an inbox file.
Messages are appended (JSONL format) for persistence and crash safety.

The team protocol supports:
  - Plan approval: workers propose a plan, coordinator approves/rejects
  - Shutdown negotiation: coordinator can request a worker to stop
  - Result aggregation: coordinator collects and synthesizes worker outputs

FULL IMPLEMENTATION NOTE:
A complete multi-agent system is too large for this notebook, but here's
the architectural skeleton. For a full working implementation

In [ ]:
@dataclass
class TeamMessage:
    """A message in the inter-agent mailbox."""
    from_agent: str
    to_agent: str
    msg_type: str   # "task", "result", "plan", "approve", "reject", "shutdown"
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class Mailbox:
    """
    JSONL-based inter-agent mailbox.

    In Claude Code, team communication uses a JSONL file per agent.
    Append-only for crash safety — if the process dies, no messages
    are lost (unlike an in-memory queue).
    """
    def __init__(self, agent_id: str, mailbox_dir: str = ".claude/mailboxes"):
        self.agent_id = agent_id
        self.path = Path(mailbox_dir) / f"{agent_id}.jsonl"
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def send(self, msg: TeamMessage):
        """Append a message to the recipient's mailbox."""
        recipient_path = self.path.parent / f"{msg.to_agent}.jsonl"
        with open(recipient_path, "a") as f:
            f.write(json.dumps(msg.__dict__) + "\n")

    def receive(self) -> list[TeamMessage]:
        """Read all messages from this agent's mailbox."""
        if not self.path.exists():
            return []
        messages = []
        for line in self.path.read_text().splitlines():
            if line.strip():
                data = json.loads(line)
                messages.append(TeamMessage(**data))
        return messages

    def clear(self):
        """Clear the mailbox after processing."""
        if self.path.exists():
            self.path.unlink()

In [ ]:
COORDINATOR_SYSTEM_PROMPT = """
You are the COORDINATOR agent managing a team of worker agents.

RULES:
- Break the user's task into subtasks and assign each to a worker agent.
- Do NOT do the work yourself — delegate via the agent tool.
- Do NOT rubber-stamp weak work. Review each worker's output critically.
- You MUST understand findings before directing follow-up work.
- Never hand off understanding to another worker.
- Synthesize all worker results into a final, coherent response.

WORKFLOW:
1. Analyze the task and create a plan (use the todo tool).
2. Spawn worker agents for each subtask (use the agent tool).
3. Review their results.
4. If a result is insufficient, spawn another agent to improve it.
5. Compile the final answer.
"""

## 3.7 MCP (Model Context Protocol)

MCP is a protocol (by Anthropic) that allows external "tool servers"
to provide tools to the agent. Instead of all tools being built-in,
MCP servers can expose tools over a standardized transport.

Architecture:

```
  Agent ←→ MCP Client ←→ MCP Server (external process)
                             └── Tools: [database_query, api_call, ...]
```

Transports:
  - stdio: the MCP server runs as a subprocess, communicates over stdin/stdout
  - HTTP+SSE: the MCP server runs as a web service

In Claude Code, MCP is configured in .claude.json:
```
  {
    "mcpServers": {
      "my-db": {
        "command": "node",
        "args": ["./mcp-db-server.js"],
        "env": {"DB_URL": "postgres://..."}
      }
    }
  }
```

At startup, Claude Code:
  1. Reads MCP config from .claude.json
  2. Spawns each MCP server as a subprocess
  3. Sends "tools/list" to discover available tools
  4. Adds those tools to the agent's tool registry
  5. When the model calls an MCP tool, routes the call to the right server

# 4) Memory

## 4.1 Architecture

Claude Code's memory system is one of its most innovative features.
It uses a 3-tier architecture:

  1. MEMORY.md — a lightweight pointer index (~150 chars/line)
    - Always loaded into context (it's tiny)
    - Contains pointers like: "Auth system: see src/auth/README.md"
    - The model maintains this file itself via the memory command
    - This is the "self-healing" part — the model curates its own memory

  2. CLAUDE.md — project/directory-level instructions
    - Loaded at startup
    - Contains conventions, architecture notes, style guides
    - Written by humans, read by the agent

  3. Session history — the conversation itself
    - Managed by compaction
    - Can be resumed across sessions

The key insight: instead of storing everything, store POINTERS to where
the information lives. The model can then file_read the actual content
on demand. This keeps the always-loaded context small.

From the VentureBeat analysis of the leak: "The architecture utilizes a
'Self-Healing Memory' system. At its core is MEMORY.md, a lightweight
index of pointers (~150 characters per line) that is perpetually loaded
into the context."

In [ ]:
class MemoryManager:
    """
    Manages the MEMORY.md file — a lightweight pointer index.

    The model can add, update, and remove entries. Each entry is a
    one-line pointer to relevant information in the codebase.

    Example MEMORY.md:
        - Auth: JWT-based, see src/auth/middleware.ts
        - DB: PostgreSQL, migrations in db/migrations/
        - Style: Use Prettier with default config
        - Testing: Jest + React Testing Library, run with `npm test`
        - API: REST endpoints in src/routes/, OpenAPI spec at docs/api.yaml
    """

    def __init__(self, memory_path: str = ".claude/MEMORY.md"):
        self.path = Path(memory_path)
        self._entries: list[str] = []
        self._load()

    def _load(self):
        if self.path.exists():
            text = self.path.read_text(encoding="utf-8")
            self._entries = [
                line.strip() for line in text.splitlines()
                if line.strip() and not line.startswith("#")
            ]

    def _save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        content = "# Agent Memory Index\n# Auto-maintained — one pointer per line\n\n"
        content += "\n".join(self._entries)
        self.path.write_text(content, encoding="utf-8")

    def add(self, entry: str) -> str:
        self._entries.append(f"- {entry}")
        self._save()
        return f"Added memory entry: {entry}"

    def remove(self, keyword: str) -> str:
        before = len(self._entries)
        self._entries = [e for e in self._entries if keyword.lower() not in e.lower()]
        removed = before - len(self._entries)
        self._save()
        return f"Removed {removed} entries matching '{keyword}'"

    def get_context_block(self) -> str:
        """Returns the memory content to inject into the system prompt."""
        if not self._entries:
            return ""
        return "# Memory Index\n" + "\n".join(self._entries)

    def list_entries(self) -> str:
        if not self._entries:
            return "Memory is empty."
        return "\n".join(self._entries)

## 4.2 Session Persistence

Claude Code persists sessions so you can resume later.
Key components:
  - Session ID (UUID per session)
  - Transcript: the full message history saved to ~/.claude/sessions/
  - Resume: load a previous session and continue the conversation
  - Export: export session as markdown for sharing

Sessions are stored as JSON files. The compacted conversation is saved,
not the raw full history (to save space).

In [ ]:
class SessionStore:
    """
    Persist and resume agent sessions.

    In Claude Code, sessions are stored under ~/.claude/sessions/{session_id}/
    with metadata, transcript, and state files.
    """
    def __init__(self, store_dir: str = ".claude/sessions"):
        self.store_dir = Path(store_dir)
        self.store_dir.mkdir(parents=True, exist_ok=True)

    def save(self, session_id: str, messages: list[dict], metadata: dict = None) -> str:
        """Save a session transcript."""
        session_dir = self.store_dir / session_id
        session_dir.mkdir(exist_ok=True)

        # Save messages (serialize ContentBlock objects)
        serializable = []
        for msg in messages:
            content = msg.get("content")
            if isinstance(content, list):
                ser_content = []
                for block in content:
                    if hasattr(block, "model_dump"):
                        ser_content.append(block.model_dump())
                    elif isinstance(block, dict):
                        ser_content.append(block)
                    else:
                        ser_content.append(str(block))
                serializable.append({**msg, "content": ser_content})
            else:
                serializable.append(msg)

        (session_dir / "transcript.json").write_text(
            json.dumps(serializable, indent=2, default=str), encoding="utf-8"
        )

        # Save metadata
        meta = {
            "session_id": session_id,
            "saved_at": datetime.now().isoformat(),
            "message_count": len(messages),
            **(metadata or {}),
        }
        (session_dir / "metadata.json").write_text(
            json.dumps(meta, indent=2), encoding="utf-8"
        )

        return str(session_dir)

    def load(self, session_id: str) -> list[dict] | None:
        """Load a session transcript. Returns None if not found."""
        transcript_path = self.store_dir / session_id / "transcript.json"
        if not transcript_path.exists():
            return None
        return json.loads(transcript_path.read_text(encoding="utf-8"))

    def list_sessions(self) -> list[dict]:
        """List all saved sessions."""
        sessions = []
        for meta_path in self.store_dir.glob("*/metadata.json"):
            sessions.append(json.loads(meta_path.read_text()))
        return sorted(sessions, key=lambda s: s.get("saved_at", ""), reverse=True)

# 5) Production Concern

Things that matter at scale but aren't core to the agent pattern:

1. PROMPT CACHING
  > Claude Code aggressively caches the system prompt. The leaked source
  has promptCacheBreakDetection.ts tracking 14 cache-break vectors,
  and "sticky latches" that prevent mode toggles from busting the cache.
  One function is annotated DANGEROUS_uncachedSystemPromptSection().
  Cache hits save ~90% on input token costs for long sessions.

2. COST TRACKING
  > Every API call's token usage is tracked (input, output, cache hits).
  Exposed via /cost command.

## 5.1 Cost Tracking

In [ ]:
class CostTracker:
    """Track API usage and costs across the session."""
    # Pricing per million tokens (approximate, Claude Sonnet 4)
    INPUT_COST_PER_M = 3.0
    OUTPUT_COST_PER_M = 15.0
    CACHE_HIT_COST_PER_M = 0.3  # 90% discount for cache hits

    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cache_hits = 0
        self.api_calls = 0

    def track(self, response):
        """Called on each API response to accumulate usage."""
        usage = response.usage
        self.total_input_tokens += usage.input_tokens
        self.total_output_tokens += usage.output_tokens
        if hasattr(usage, "cache_read_input_tokens"):
            self.total_cache_hits += usage.cache_read_input_tokens or 0
        self.api_calls += 1

    def report(self) -> str:
        input_cost = (self.total_input_tokens / 1_000_000) * self.INPUT_COST_PER_M
        output_cost = (self.total_output_tokens / 1_000_000) * self.OUTPUT_COST_PER_M
        cache_savings = (self.total_cache_hits / 1_000_000) * (self.INPUT_COST_PER_M - self.CACHE_HIT_COST_PER_M)
        total = input_cost + output_cost
        return (
            f"API calls: {self.api_calls}\n"
            f"Input tokens: {self.total_input_tokens:,}\n"
            f"Output tokens: {self.total_output_tokens:,}\n"
            f"Cache hits: {self.total_cache_hits:,}\n"
            f"Estimated cost: ${total:.4f}\n"
            f"Cache savings: ${cache_savings:.4f}"
        )

## 5.2 HOOKS SYSTEM

Claude Code supports PreToolUse and PostToolUse hooks — user-defined
scripts that run before/after each tool call.

Configured in .claude.json:
```
  {
    "hooks": {
      "PreToolUse": [{"matcher": "bash", "command": "echo $TOOL_INPUT | jq .command"}],
      "PostToolUse": [{"matcher": "*", "command": "log-tool-call.sh"}]
    }
  }
```

Hooks can: approve, deny, modify, or log tool calls.

NOTE: The leaked source shows hooks are parsed but not fully executed
in the Rust port. The TypeScript version has full hook support.

**ANTI-DISTILLATION**
> Claude Code sends anti_distillation: ['fake_tools'] in API requests.
This tells the server to inject DECOY tool definitions into the prompt.
If someone records API traffic to train a competing model, the fake
tools pollute that training data. Gated behind a feature flag.

**FRUSTRATION DETECTION**
> Claude Code uses regex to detect user frustration:
  patterns like "this is broken", "why doesn't this work", etc.
When detected, the agent adjusts its behavior to be more careful
and explicit about its reasoning.

# 6) Agent Harness

In [ ]:
class AgentHarness:
    """
    A complete agent harness combining all the pieces from this notebook.

    This is a minimal but functional equivalent of Claude Code's core loop
    with tools, permissions, planning, memory, and session management.
    """

    def __init__(
        self,
        project_root: str = ".",
        permission_mode: str = "auto",
        session_id: str = None,
    ):
        self.project_root = project_root
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.messages: list[dict] = []
        self.tool_call_count = 0

        # ── Initialize all subsystems ─────────────────────────────────
        self.permission_policy = PermissionPolicy(mode=permission_mode)
        self.todo_manager = TodoManager()
        self.memory_manager = MemoryManager()
        self.session_store = SessionStore()
        self.cost_tracker = CostTracker()

        # ── Build tool registry ───────────────────────────────────────
        self.tool_schemas = CORE_TOOL_SCHEMAS + [TODO_TOOL_SCHEMA, AGENT_TOOL_SCHEMA]
        self.tool_handlers = {
            **CORE_TOOL_HANDLERS,
            "todo": todo_write_handler,
            "agent": lambda task: agent_tool(
                task=task,
                parent_system=self._build_system_prompt(),
                tools=CORE_TOOL_SCHEMAS,
                tool_handlers=CORE_TOOL_HANDLERS,
            ),
        }

        # ── Assemble system prompt ────────────────────────────────────
        self.system_prompt = self._build_system_prompt()

    def _build_system_prompt(self) -> str:
        """Assemble the full system prompt from all sources."""
        prompt = assemble_system_prompt(
            project_root=self.project_root,
            permission_mode=self.permission_policy.mode,
        )
        # Inject memory
        memory_block = self.memory_manager.get_context_block()
        if memory_block:
            prompt += f"\n\n{memory_block}"
        return prompt

    def _on_tool_call(self, tool_name: str, tool_input: dict):
        """Hook called before each tool execution."""
        self.tool_call_count += 1

        # Permission check
        allowed, reason = self.permission_policy.check_tool(tool_name, tool_input)
        if not allowed:
            print(f"  🚫 [{tool_name}] BLOCKED: {reason}")
        else:
            print(f"  🔧 [{tool_name}] {json.dumps(tool_input)[:80]}")

        # Nag reminder
        nag = self.todo_manager.get_nag_reminder(self.tool_call_count)
        if nag:
            print(f"  📋 {nag}")

    def _on_response(self, response):
        """Hook called after each LLM response."""
        self.cost_tracker.track(response)

    def run(self, user_message: str) -> str:
        """
        Send a message to the agent and get a response.

        This is the main entry point — equivalent to typing a message
        in Claude Code's terminal REPL.
        """
        self.messages.append({"role": "user", "content": user_message})

        # Run the agent loop
        self.messages = agent_loop(
            messages=self.messages,
            system=self.system_prompt,
            tools=self.tool_schemas,
            tool_handlers=self.tool_handlers,
            on_tool_call=self._on_tool_call,
            on_response=self._on_response,
        )

        # Context compaction check
        self.messages = compact_messages(
            self.messages, self.system_prompt, max_tokens=80000
        )

        # Extract final text
        last = self.messages[-1]
        if last["role"] == "assistant":
            texts = []
            for block in last["content"]:
                if hasattr(block, "text"):
                    texts.append(block.text)
            return "\n".join(texts) if texts else "(no text output)"

        return "(unexpected state)"

    def save_session(self):
        """Persist the current session."""
        path = self.session_store.save(
            self.session_id,
            self.messages,
            metadata={"tool_calls": self.tool_call_count},
        )
        print(f"  💾 Session saved to {path}")

    def show_cost(self):
        """Print cost summary."""
        print(self.cost_tracker.report())

In [ ]:
print("\nInitializing agent harness...")
agent = AgentHarness(permission_mode="dangerFullAccess")
print(f"Session: {agent.session_id}")
print(f"Tools: {[t['name'] for t in agent.tool_schemas]}")
print(f"System prompt: {len(agent.system_prompt)} chars\n")

# Single-turn demo
response = agent.run(
    "List the Python files in the current directory and tell me what this project is about."
)
print(f"\n🤖 Agent response:\n{response}")

# Show cost
print(f"\n💰 Cost report:")
agent.show_cost()

# Save session
agent.save_session()


Initializing agent harness...
Session: 24778649
Tools: ['bash', 'file_read', 'file_write', 'glob', 'grep', 'todo', 'agent']
System prompt: 746 chars

  🔧 [glob] {"pattern": "**/*.py"}
  🔧 [bash] {"command": "ls -la /content && cat /content/README* 2>/dev/null || echo \"No RE

🤖 Agent response:
The current directory (`/content`) appears to be essentially **empty in terms of Python files and project files**. Here's what was found:

- **No Python files (`.py`)** exist anywhere in the directory tree.
- **No README** or documentation files are present.
- The directory contains only:
  - `.claude/` — configuration for the Claude agent environment.
  - `.config/` — general system/tool configuration.
  - `sample_data/` — a folder with some sample data (likely default content from the environment, e.g., Google Colab).

### Summary
This does **not appear to be an established project** yet. The `/content` directory looks like a **fresh or empty environment** (typical of a Google Colab or similar